# SAT-12 · ALL paper results in one run = notebook 04 + notebook 09  (CPU, ~30-45 min)

Use this instead of running 04 and 09 separately. **Attach:** everything listed for 04 (sat-ext, sat-splits, sat-splits-extra,
sat-splits-sky, satelite-coco, satelite-native, skysealand-v14, every `sat-results-*` incl. 03). **Accelerator:** None.
**Internet:** ON (09 part installs `ensemble-boxes`). Output: `results/paper/` with every table (T1-T16, D1-D4) and figure.

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `satelite-coco` - Roboflow COCO export, 640x640 (or download: universe.roboflow.com/rifat4018/aerial-object-detection-u3f5u): https://www.kaggle.com/datasets/rifat4018/satelite-coco
* `satelite-native` - Roboflow COCO export, native size, no augmentation: https://www.kaggle.com/datasets/rifat4018/satelite-native
* `sat-splits` - output of notebook 01: https://www.kaggle.com/datasets/rifat4018/sat-splits
* `sat-splits-sky` - output of notebook 01b: https://www.kaggle.com/datasets/rifat4018/sat-splits-sky
* `sat-splits-extra` - output of notebook 06 (optional): https://www.kaggle.com/datasets/rifat4018/sat-splits-extra
* `skysealand-v14` - SkySeaLand (public, Mendeley Data): https://data.mendeley.com/datasets/d42n3cp86p
* Check the printed input tree in the next cell before running the rest.

In [ ]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

In [ ]:
utils.warn_if_gpu_wasted()
import report, plans, dataset as ds
runs = report.collect_runs()
df = report.long_table(runs)
os.makedirs(report.PAPER_DIR, exist_ok=True)
df.drop(columns=["folder"]).to_csv(os.path.join(report.PAPER_DIR, "all_runs_long.csv"), index=False)
expected = set(plans.expected_run_ids())
optional = set(plans.expected_run_ids(include_optional=True)) - expected
print("phase-2 / optional runs found:", len(optional & set(runs)), "of", len(optional),
      "| not found:", sorted(optional - set(runs))[:40])
missing = sorted(expected - set(runs))
print("expected runs:", len(expected), "| found:", len(expected & set(runs)), "| MISSING:", missing or "none")
print("extra runs (legacy etc.):", sorted(set(runs) - expected))
for rid, (d, _) in runs.items():
    if "engine_check" in d:
        print("pycocotools vs internal AP (should be equal):", rid, d["engine_check"])

In [ ]:
t1 = report.table_main(df, "grouped")
t2 = report.table_sahi(df, "grouped")
t3 = report.table_bootstrap(df, "grouped")
t4 = report.table_per_class(df, "grouped")
t5 = report.table_orig_vs_grouped(df)
if (df.split == "orig").any():
    report.table_sahi(df, "orig")
if (df.split == "sky_grouped").any():           # second dataset (one seed)
    report.table_main(df, "sky_grouped"); report.table_sahi(df, "sky_grouped"); report.table_per_class(df, "sky_grouped")
    report.fig_pareto(df, "sky_grouped"); report.fig_size_recall(df, "sky_grouped")
t6 = report.table_transfer(df)
report.table_splits_compare(df, ["grouped_1copy", "grouped"], ["YOLO11s", "YOLO11m"], "T7a_offline_augmentation",
    "Offline augmentation ablation: train on 1 vs all 3 Roboflow augmented copies per source (same test images).")
report.table_splits_compare(df, [f"grouped_lc{int(f * 100)}" for f in cfg.LEARNING_CURVE] + ["grouped"], ["YOLO11s", "YOLO11m"],
    "T7b_learning_curve", "Learning curve: share of the training ground-area groups used (same test images).")
print(report.fig_learning_curve(df))
report.table_native(df)
abl = report.collect_ablation()
if len(abl):
    report.table_sahi_ablation(abl, "grouped"); report.table_sahi_ablation(abl, "sky_grouped")
report.table_external(df)
report.table_efficiency(df, "grouped")

In [ ]:
for f in [report.fig_pareto(df, "grouped"), report.fig_size_recall(df, "grouped"), report.fig_extra_dets(df, "grouped")]:
    print(f)
info = ds.materialize("grouped")
print(report.fig_qualitative(df, info, split="grouped"))

In [ ]:
try:
    from IPython.display import Image as _I, display
    for f in sorted(os.listdir(report.PAPER_DIR)):
        if f.endswith(".png"):
            print(f); display(_I(os.path.join(report.PAPER_DIR, f), width=900))
except ImportError:
    print(sorted(os.listdir(report.PAPER_DIR)))

In [ ]:
utils.warn_if_gpu_wasted()
utils.pip_install("ensemble-boxes")
import report, analysis
runs = report.collect_runs()
df = report.long_table(runs)
os.makedirs(report.PAPER_DIR, exist_ok=True)

In [ ]:
err = analysis.table_errors(df, "grouped")
print(analysis.fig_errors(err, "grouped"))
if (df.split == "sky_grouped").any():
    analysis.fig_errors(analysis.table_errors(df, "sky_grouped"), "sky_grouped")

In [ ]:
analysis.table_mcnemar(df, "grouped", "standard")
analysis.table_mcnemar_modes(df, "grouped")
if "sky_grouped" in set(df.split):
    analysis.table_mcnemar(df, "sky_grouped", "standard"); analysis.table_mcnemar_modes(df, "sky_grouped")
# native resolution (07J-07L): native_tiles has the SAME val/test images and ids as native_grouped -> one paired table
nat = df[df.split.isin(["native_grouped", "native_tiles"])].copy()
if len(nat):
    nat["model"] = nat.model.astype(str) + np.where(nat.split == "native_tiles", " (tiles)", "")
    nat["split"] = "native_grouped"
    analysis.fig_errors(analysis.table_errors(nat, "native_grouped", models=sorted(set(nat.model))), "native_grouped")
    analysis.table_mcnemar(nat, "native_grouped", "standard", models=sorted(set(nat.model)))
    analysis.table_mcnemar_modes(nat, "native_grouped", models=sorted(set(nat.model)))
    analysis.table_ensembles(df, "native_grouped")     # WBF of seeds / architectures at native resolution
else:
    print("no native runs attached (07J-07L) -> native McNemar skipped")

In [ ]:
analysis.table_ensembles(df, "grouped")
import shutil
print(shutil.make_archive(os.path.join(cfg.WORK_DIR, "paper_analysis"), "zip", report.PAPER_DIR))